<a href="https://colab.research.google.com/github/Ahirvoas/Training-Day2/blob/main/Tutorial_day2-Anomaly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Wind Turbine Modeling Workshop

**How to use**
- Work through the notebook from top to bottom.
- Complete the `# TODO` parts. Hints are available inside `# Solution` sections (commented).
- Run cells incrementally and inspect intermediate outputs.


In [ ]:
!pip install gdown &> /dev/null
!gdown https://drive.google.com/uc?id=1h4pa6sSFbgaAi8zHObwtQIDVB8pBXoG-

In [ ]:
!ls

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, GridSearchCV
from imblearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

---

## Context and quick theory

Wind turbines are instrumented with multiple sensors (vibration, temperature, rotational speed, power, etc.).  
Anomalies in sensor readings may indicate faults, wear, or external events (e.g., icing, sudden gusts).  
Detecting anomalies early helps scheduling maintenance and preventing failures.

**Common approaches**:
- Simple statistical thresholds (mean ± k * std)  
- Time-series decomposition, rolling statistics  
- Machine learning models for anomaly detection (Isolation Forest, One-Class SVM, Autoencoders)

In this notebook, you will explore the dataset, preprocess signals, and run the anomaly detection pipeline.


## Part 1 — Data Loading & Exploration

**Goal:** Load the sensor dataset, examine time index, basic statistics, and plot example signals.

### Exercise
- Complete the `# EXERCISE` cell following this markdown to load the dataset and show basic stats.

In [ ]:
# EXERCISE: Load the dataset and show basic statistics (head, describe, time index check).
# TODO: Complete the code below


### Exercise
- Implement preprocessing (scaling).

In [ ]:
# EXERCISE: scale the data using StandardScaler.
# TODO: Complete the code below


In [ ]:
df['Fault'].unique()

The fault dataset catalogs various fault types or modes that can occur in wind turbines. Specifically, it includes three types of faults:

* mf: Mains Failure Fault
* af: Timeout Warning Message - Malfunction Air Cooling
* ef: Excitation Error - Overvoltage DC-Link

This information is essential for diagnosing and understanding the different failure mechanisms that may affect turbine performance and reliability.

### Exercise
- Stratified split to maintain anomaly ratio using train_test_split function

In [ ]:
X = df.iloc[:,:-1]
y = df.iloc[:,-1]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

- Plot anomalies based on scatter plot make a color difference between train and test set

In [ ]:
# EXERCISE: Plot anomalies based on scatter plot from tutorial in the tutorial from Day 1.
# TODO: Complete the code below


# Your code here

## Part 3 — Anomaly Detection

**Goal:** Use the anomaly detection algorithms. Train the model and obtain anomaly scores.

### Exercise
- Train a model and compute anomaly scores.

In [ ]:
# Make pipeline of scaling, and classifier named knn_pipe

In [ ]:
# Define multiple scoring metrics [accuracy, recall_macro, f1_macro]

### Stratified K-Fold cross validation

When we train machine learning models, we need to **evaluate** how well they generalize to **unseen data**.  
A common way to do this is **K-Fold Cross Validation (CV)** — but **Stratified K-Fold** is a **special, improved version** for **classification problems**.

---

#### Regular K-Fold Cross Validation

In **K-Fold CV**, we:
1. Split the dataset into **K equal parts (folds)**.
2. Train the model on **K−1 folds**, and test it on the **remaining one**.
3. Repeat this **K times**, each time using a different fold as the test set.
4. Average the results to get the **final model performance**.

✅ **Advantage:** Efficient use of all data for both training and testing.  
⚠️ **Problem:** If the dataset is **imbalanced** (e.g., 90% class A, 10% class B), some folds might contain **almost no samples from class B**, making evaluation unreliable.

---

#### Stratified K-Fold Cross Validation

**Stratified K-Fold** fixes this problem by ensuring that **each fold has approximately the same class proportion** as the overall dataset.

So, if your dataset has 70% of class 0 and 30% of class 1:
- Each fold will also contain ~70% class 0 and ~30% class 1.

✅ **Better for classification problems**  
✅ **Keeps class balance**  
✅ **More stable and fair evaluation**

In [ ]:
# Stratified K-Fold
stratkfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# Hyperparameter tuning
param_grid = {'kneighborsclassifier__n_neighbors': range(3, 11)}
grid_search = GridSearchCV(knn_pipe, param_grid, cv=stratkfold, scoring='f1_macro', n_jobs=-1)
grid_search.fit(X_train, y_train)


# Best model
best_knn_pipe = grid_search.best_estimator_


In [ ]:
# Fit best model best_knn_pipe on full training set


In [ ]:
# Predict on test set


In [ ]:
# Confusion matrix of test set
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap=plt.cm.Greens)
plt.show()

---

## Conclusion & Next Steps

- Discuss the limitations of the approach used in this notebook (false positives/negatives, concept drift, sensor faults that mimic anomalies).  
- Extensions for further work: time-series specific models (LSTM autoencoders), ensemble detection, data augmentation, root-cause analysis, explainability.

**Instructor note:** Solution hints were provided as commented blocks in exercise cells. Students should uncomment the Solution block only after attempting the exercise.



## Appendix — Understanding the Confusion Matrix

When evaluating an anomaly detection model, especially when ground truth labels are available, 
it is important to understand how well the model distinguishes **normal** versus **anomalous** cases.

The **confusion matrix** is a 2×2 table that compares predicted labels with actual labels:

|                     | Predicted Normal | Predicted Anomalous |
|---------------------:|:----------------:|:-------------------:|
| **Actual Normal**    | True Negative (TN) | False Positive (FP) |
| **Actual Anomalous** | False Negative (FN) | True Positive (TP) |

- **True Positive (TP)** — the model correctly detected an actual anomaly.  
- **False Positive (FP)** — the model flagged a normal sample as an anomaly (false alarm).  
- **True Negative (TN)** — the model correctly classified normal behavior.  
- **False Negative (FN)** — the model missed an anomaly (undetected fault).  

From this matrix, several performance metrics can be derived:

- **Precision** = TP / (TP + FP): among all detected anomalies, how many were correct?  
- **Recall** = TP / (TP + FN): among all real anomalies, how many were detected?  
- **F1-score** = 2 × (Precision × Recall) / (Precision + Recall): harmonic mean balancing both.

In anomaly detection, recall is often prioritized — we prefer to detect as many anomalies as possible, 
even at the cost of some false alarms. However, in industrial systems (like wind turbines), 
too many false positives can lead to unnecessary maintenance actions, so balance is key.
